# Lesson 08 Lab — Pointer Arithmetic and Tensor Layout

**Puzzle:** When logical indices, physical strides, and transposed views change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates logical indices, physical strides, and transposed views and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A logical row and column become a physical address only after multiplying by strides. Writing that equation explicitly lets one Triton kernel accept contiguous, transposed, sliced, or padded views without pretending they share storage order.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["logical indices, physical strides, and transposed views"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Testing only contiguous tensors can hide a pointer formula that is correct by accident.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 8
LESSON_TITLE = 'Pointer Arithmetic and Tensor Layout'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260821
}


## 5. Freeze the experiment

**Experiment:** Copy a transposed non-contiguous view into a contiguous output with explicit row and column strides.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.02731199935078621,
  "secondary": 461.3099113755403,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "input_stride": [
      1,
      2048
    ],
    "output_stride": [
      769,
      1
    ],
    "samples_ms": [
      0.03488000109791756,
      0.031488001346588135,
      0.028736000880599022,
      0.028416000306606293,
      0.027424000203609467,
      0.02672000043094158,
      0.030559999868273735,
      0.027807999402284622,
      0.02707199938595295,
      0.02579200081527233,
      0.02707199938595295,
      0.027327999472618103,
      0.027264000847935677,
      0.02723200060427189,
      0.026559999212622643,
      0.026559999212622643,
      0.027295999228954315,
      0.028031999245285988,
      0.027327999472618103,
      0.027295999228954315
    ]
  }
}
Explicit stride arithmetic copied a transposed 2048x769 view correctly at 461.3 requested GB/s.


## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Kernel median | 0.0273 ms |
| Requested bandwidth | 461.3099 |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

Explicit stride arithmetic copied a transposed 2048x769 view correctly at 461.3 requested GB/s.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Make strides part of the public operator contract whenever layout is not fixed by construction.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 8,
  "title": "Pointer Arithmetic and Tensor Layout",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260821
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.02731199935078621,
    "secondary": 461.3099113755403,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "input_stride": [
        1,
        2048
      ],
      "output_stride": [
        769,
        1
      ],
      "samples_ms": [
        0.03488000109791756,
        0.031488001346588135,
        0.028736000880599022,
        0.028416000306606293,
        0.027424000203609467,
        0.02672000043094158,
        0.030559999868273735,
        0.027807999402284622,
        0.02707199938595295,
        0.02579200081527233,
        0.0270719

## 10. Make the bounded decision

> Make strides part of the public operator contract whenever layout is not fixed by construction.

**Failure analysis:** Testing only contiguous tensors can hide a pointer formula that is correct by accident.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
